# Что меняем в датасете после EDA

## 1.1. Основной train frame

Теперь для обучения используем:
```
match_dataset_1994_2022_with_text
```

а не полный исторический:
```
match_dataset 1930-2022
```
Полный historical dataset 1930–2022 остается полезен для:
```
- EDA
- historical graph
- team history
- visualization
- long-term football ecosystem
```
Но supervised outcome prediction лучше строить на:
```
1994–2022
```
потому что FIFA ranking features доступны только с этой эпохи.

In [3]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight


In [7]:
PROJECT_ROOT = Path("SNA").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TEXT_PROCESSED_DIR = PROCESSED_DIR / "text"

MATCH_DATASET_TEXT_PATH = PROCESSED_DIR / "match_dataset_1994_2022_with_text.parquet"
MATCH_DATASET_TEXT_CSV_PATH = PROCESSED_DIR / "match_dataset_1994_2022_with_text.csv"

PRE_MATCH_ARTICLES_PATH = TEXT_PROCESSED_DIR / "pre_match_articles.parquet"
PRE_MATCH_ARTICLES_CSV_PATH = TEXT_PROCESSED_DIR / "pre_match_articles.csv"

MODEL_READY_PATH = PROCESSED_DIR / "match_dataset_model_ready.parquet"

FEATURE_GROUPS_PATH = PROCESSED_DIR / "feature_groups.json"
CLASS_WEIGHTS_PATH = PROCESSED_DIR / "class_weights.json"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TEXT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [8]:
def load_existing_table(parquet_path, csv_path=None):
    parquet_path = Path(parquet_path)

    if parquet_path.exists():
        return pd.read_parquet(parquet_path)

    if csv_path is not None:
        csv_path = Path(csv_path)
        if csv_path.exists():
            return pd.read_csv(csv_path)

    raise FileNotFoundError(f"No file found: {parquet_path} or {csv_path}")


def save_table(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.suffix.lower() == ".parquet":
        try:
            df.to_parquet(path, index=index)
            print(f"[OK] Saved parquet: {path}")
        except Exception as e:
            fallback = path.with_suffix(".csv")
            print(f"[WARN] Parquet failed: {e}")
            print(f"[WARN] Saving CSV instead: {fallback}")
            df.to_csv(fallback, index=index)
    elif path.suffix.lower() == ".csv":
        df.to_csv(path, index=index)
        print(f"[OK] Saved csv: {path}")
    else:
        raise ValueError(f"Unsupported suffix: {path.suffix}")


In [9]:
df = load_existing_table(
    MATCH_DATASET_TEXT_PATH,
    MATCH_DATASET_TEXT_CSV_PATH,
)

df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce")

print(df.shape)
display(df.head())
display(df["tournament_year"].value_counts().sort_index())
display(df["result"].value_counts(normalize=True))


(500, 109)


,tournament_year,match_date,stage,home_team_name,away_team_name,home_team_norm,away_team_norm,home_score,away_score,stadium_name,...,away_tournament_goals_against_before,away_tournament_goal_diff_before,away_rest_days,away_first_match_in_tournament,text_available,text_count,combined_pre_match_text,sources,latest_text_time,min_hours_before_match
0,1994,1994-06-17,Group stage,Germany,Bolivia,germany,bolivia,1,0,"Soldier Field, Chicago",...,0,0,NaN,True,0,0,,,NaT,NaN
1,1994,1994-06-17,Group stage,Spain,Korea Republic,spain,south korea,2,2,"Cotton Bowl, Dallas",...,0,0,NaN,True,0,0,,,NaT,NaN
2,1994,1994-06-18,Group stage,Colombia,Romania,colombia,romania,1,3,"Rose Bowl, Los Angeles",...,0,0,NaN,True,0,0,,,NaT,NaN
3,1994,1994-06-18,Group stage,Italy,Republic of Ireland,italy,ireland,0,1,"Giants Stadium, New York/New Jersey",...,0,0,NaN,True,0,0,,,NaT,NaN
4,1994,1994-06-18,Group stage,United States,Switzerland,united states,switzerland,1,1,"Pontiac Silverdome, Detroit",...,0,0,NaN,True,0,0,,,NaT,NaN


tournament_year
1994    52
1998    64
2002    64
2006    64
2010    64
2014    64
2018    64
2022    64
Name: count, dtype: int64

result
home_win    0.438
away_win    0.326
draw        0.236
Name: proportion, dtype: float64

In [10]:
try:
    pre_articles = load_existing_table(
        PRE_MATCH_ARTICLES_PATH,
        PRE_MATCH_ARTICLES_CSV_PATH,
    )
    print(pre_articles.shape)
    display(pre_articles.head())
except FileNotFoundError:
    pre_articles = pd.DataFrame()
    print("[WARN] pre_match_articles not found. Text features will be based on existing match-level fields only.")


(994, 26)


,source,url,title,published_at,section_name,type,language,raw_text,snippet,byline,...,collector,collected_at,text_type,is_pre_match,leakage_risk_flag,combined_raw,clean_text,text_id,official_match_date_utc,hours_before_match
0,guardian,https://www.theguardian.com/football/2014/jun/...,World Cup 2014: Algeria’s Vahid Halilhodzic pr...,2014-06-25 13:42:00+00:00,Football,article,en,Vahid Halilhodzic had just seen Algeria produc...,After delivering their first finals win since ...,Nick Ames,...,guardian_open_platform,2026-05-22T21:31:52.140172+00:00,None,True,False,World Cup 2014: Algeria’s Vahid Halilhodzic pr...,World Cup 2014: Algeria’s Vahid Halilhodzic pr...,text_f413e65034bb,2014-06-26 00:00:00+00:00,10.300000
1,guardian,https://www.theguardian.com/football/2014/jun/...,South Korea v Algeria: World Cup 2014 - as it ...,2014-06-22 21:01:34+00:00,Football,liveblog,en,Who expected a classy attacking display from A...,<b>As it happened:</b> South Korea and Algeria...,Toby Moses,...,guardian_open_platform,2026-05-22T21:31:52.140141+00:00,None,True,False,South Korea v Algeria: World Cup 2014 - as it ...,South Korea v Algeria: World Cup 2014 - as it ...,text_04074cd9e5a6,2014-06-26 00:00:00+00:00,74.973889
2,guardian,https://www.theguardian.com/football/2014/jun/...,Belgium v Russia - World Cup 2014 as it happened,2014-06-22 17:53:34+00:00,Football,liveblog,en,"Well, that was utterly dire for about 85 minut...",<b>Minute-by-minute report:</b> Belgium are th...,Nick Miller,...,guardian_open_platform,2026-05-22T21:31:53.505914+00:00,None,True,False,Belgium v Russia - World Cup 2014 as it happen...,Belgium v Russia - World Cup 2014 as it happen...,text_45695f95ae24,2014-06-26 00:00:00+00:00,78.107222
3,guardian,https://www.theguardian.com/football/video/201...,World Cup Show 2014: day 11 previews – video,2014-06-22 12:00:00+00:00,Football,video,en,<p>Nat Coombs is joined by Guardian journalist...,<p>Nat Coombs is joined by Guardian journalist...,NaN,...,guardian_open_platform,2026-05-22T21:31:52.140061+00:00,None,True,False,World Cup Show 2014: day 11 previews – video. ...,World Cup Show 2014: day 11 previews – video. ...,text_9b7aefd504b6,2014-06-26 00:00:00+00:00,84.000000
4,guardian,https://www.theguardian.com/football/live/2022...,Croatia 4-1 Canada: World Cup 2022 – as it hap...,2022-11-27 18:31:27+00:00,Football,liveblog,en,Bryan Graham was at the match today and here’s...,The 2018 finalists came back from an early def...,Beau Dure,...,guardian_open_platform,2026-05-22T21:08:50.833521+00:00,None,True,False,Croatia 4-1 Canada: World Cup 2022 – as it hap...,Croatia 4-1 Canada: World Cup 2022 – as it hap...,text_3da7fbb86b9e,2022-11-28 00:00:00+00:00,5.475833


In [11]:
def filter_articles_min_hours_before_match(articles, min_hours=6):
    """
    Оставляет статьи, опубликованные минимум min_hours до матча.
    """
    if articles.empty:
        return articles.copy()

    out = articles.copy()

    if "published_at" in out.columns:
        out["published_at"] = pd.to_datetime(out["published_at"], errors="coerce", utc=True)

    if "official_match_date_utc" in out.columns:
        out["official_match_date_utc"] = pd.to_datetime(out["official_match_date_utc"], errors="coerce", utc=True)
    elif "match_date" in out.columns:
        out["official_match_date_utc"] = pd.to_datetime(out["match_date"], errors="coerce", utc=True)
    else:
        raise KeyError("No match date column found in articles.")

    if "hours_before_match" not in out.columns:
        out["hours_before_match"] = (
            out["official_match_date_utc"] - out["published_at"]
        ).dt.total_seconds() / 3600

    out = out[
        (out["hours_before_match"] >= min_hours)
        & (out["published_at"].notna())
        & (out["official_match_date_utc"].notna())
    ].copy()

    out["min_6h_safe_text_flag"] = True

    return out


pre_articles_6h = filter_articles_min_hours_before_match(pre_articles, min_hours=6)

print("Before 6h filter:", pre_articles.shape)
print("After 6h filter:", pre_articles_6h.shape)

if not pre_articles_6h.empty:
    display(pre_articles_6h["tournament_year"].value_counts().sort_index())


Before 6h filter: (994, 26)
After 6h filter: (867, 27)


tournament_year
1994      1
1998      1
2002    121
2006    111
2010    152
2014    219
2018    116
2022    146
Name: count, dtype: int64

In [13]:
save_table(
    pre_articles_6h,
    TEXT_PROCESSED_DIR / "pre_match_articles_6h.parquet",
)
save_table(
    pre_articles_6h,
    TEXT_PROCESSED_DIR / "pre_match_articles_6h.csv",
)


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/text/pre_match_articles_6h.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/text/pre_match_articles_6h.csv


In [14]:
def aggregate_pre_match_texts_to_match_level(
    pre_match_articles,
    matches,
    max_articles_per_match=5,
    max_chars_per_article=3000,
):
    """
    Агрегирует статьи до уровня матча.
    Берет последние max_articles_per_match safe-статей перед матчем.
    """
    matches_base_cols = [
        "match_id",
        "tournament_year",
        "match_date",
        "home_team_id",
        "away_team_id",
        "home_team_name",
        "away_team_name",
    ]

    matches_base_cols = [c for c in matches_base_cols if c in matches.columns]

    matches_base = matches[matches_base_cols].copy()

    if pre_match_articles.empty:
        out = matches_base.copy()
        out["text_available"] = 0
        out["text_count"] = 0
        out["log_text_count"] = 0.0
        out["combined_pre_match_text"] = ""
        out["sources"] = ""
        out["latest_text_time"] = pd.NaT
        out["earliest_text_time"] = pd.NaT
        out["min_hours_before_match"] = np.nan
        out["max_hours_before_match"] = np.nan
        out["guardian_text_flag"] = 0
        return out

    articles = pre_match_articles.copy()

    articles["published_at"] = pd.to_datetime(
        articles["published_at"],
        errors="coerce",
        utc=True,
    )

    if "clean_text" not in articles.columns:
        if "raw_text" in articles.columns:
            articles["clean_text"] = articles["raw_text"].fillna("").astype(str)
        else:
            articles["clean_text"] = ""

    articles = articles.sort_values(
        ["match_id", "published_at"],
        ascending=[True, False],
    )

    articles["rank_within_match"] = articles.groupby("match_id").cumcount() + 1

    articles = articles[
        articles["rank_within_match"] <= max_articles_per_match
    ].copy()

    articles["clean_text_short"] = (
        articles["clean_text"]
        .fillna("")
        .astype(str)
        .str.slice(0, max_chars_per_article)
    )

    grouped = (
        articles.groupby("match_id")
        .agg(
            text_count=("text_id", "count"),
            sources=("source", lambda s: ",".join(sorted(set([str(x) for x in s if pd.notna(x)])))),
            latest_text_time=("published_at", "max"),
            earliest_text_time=("published_at", "min"),
            min_hours_before_match=("hours_before_match", "min"),
            max_hours_before_match=("hours_before_match", "max"),
            combined_pre_match_text=(
                "clean_text_short",
                lambda texts: "\n\n[ARTICLE_SEP]\n\n".join(
                    [t for t in texts if isinstance(t, str) and len(t) > 0]
                ),
            ),
        )
        .reset_index()
    )

    out = matches_base.merge(grouped, on="match_id", how="left")

    out["text_count"] = out["text_count"].fillna(0).astype(int)
    out["text_available"] = (out["text_count"] > 0).astype(int)
    out["log_text_count"] = np.log1p(out["text_count"])

    out["sources"] = out["sources"].fillna("")
    out["combined_pre_match_text"] = out["combined_pre_match_text"].fillna("")
    out["guardian_text_flag"] = out["sources"].str.contains("guardian", case=False, na=False).astype(int)

    return out


match_texts_6h = aggregate_pre_match_texts_to_match_level(
    pre_articles_6h,
    df,
    max_articles_per_match=5,
    max_chars_per_article=3000,
)

print(match_texts_6h.shape)
display(match_texts_6h.head())
display(
    match_texts_6h.groupby("tournament_year")
    .agg(
        matches=("match_id", "count"),
        matches_with_text=("text_available", "sum"),
        avg_text_count=("text_count", "mean"),
    )
)


(500, 17)


,match_id,tournament_year,match_date,home_team_id,away_team_id,home_team_name,away_team_name,text_count,sources,latest_text_time,earliest_text_time,min_hours_before_match,max_hours_before_match,combined_pre_match_text,text_available,log_text_count,guardian_text_flag
0,match_24ab401684d2,1994,1994-06-17,team_28ef36b35ae8,team_76781be1c79d,Germany,Bolivia,0,,NaT,NaT,NaN,NaN,,0,0.0,0
1,match_38ef6f79f047,1994,1994-06-17,team_1747d4b2dd4b,team_3c6cfbc2415a,Spain,Korea Republic,0,,NaT,NaT,NaN,NaN,,0,0.0,0
2,match_ebe85d2cc918,1994,1994-06-18,team_25446782e2cc,team_89055f975d7d,Colombia,Romania,0,,NaT,NaT,NaN,NaN,,0,0.0,0
3,match_f0b250615ce7,1994,1994-06-18,team_bab4ac375ada,team_a66f852274a4,Italy,Republic of Ireland,0,,NaT,NaT,NaN,NaN,,0,0.0,0
4,match_fd7cb15508dc,1994,1994-06-18,team_152649df347e,team_0d9c575f4f02,United States,Switzerland,0,,NaT,NaT,NaN,NaN,,0,0.0,0


,matches,matches_with_text,avg_text_count
tournament_year,,,
1994,52,1,0.019231
1998,64,1,0.015625
2002,64,57,1.843750
2006,64,39,1.703125
2010,64,61,2.343750
2014,64,62,3.328125
2018,64,51,1.812500
2022,64,49,2.109375


In [15]:
save_table(
    match_texts_6h,
    PROCESSED_DIR / "match_texts_6h.parquet",
)
save_table(
    match_texts_6h,
    PROCESSED_DIR / "match_texts_6h.csv",
)

[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/match_texts_6h.parquet
[OK] Saved csv: /Users/gaperov/Documents/University/SNA/data/processed/match_texts_6h.csv


In [17]:
text_cols = [
    "text_available",
    "text_count",
    "log_text_count",
    "combined_pre_match_text",
    "sources",
    "latest_text_time",
    "earliest_text_time",
    "min_hours_before_match",
    "max_hours_before_match",
    "guardian_text_flag",
]

df_model = df.drop(columns=text_cols, errors="ignore").copy()

merge_cols = ["match_id"] + [c for c in text_cols if c in match_texts_6h.columns]

df_model = df_model.merge(
    match_texts_6h[merge_cols],
    on="match_id",
    how="left",
)

df_model["text_available"] = df_model["text_available"].fillna(0).astype(int)
df_model["text_count"] = df_model["text_count"].fillna(0).astype(int)
df_model["log_text_count"] = np.log1p(df_model["text_count"])
df_model["combined_pre_match_text"] = df_model["combined_pre_match_text"].fillna("")
df_model["sources"] = df_model["sources"].fillna("")
df_model["guardian_text_flag"] = df_model["guardian_text_flag"].fillna(0).astype(int)

print(df_model.shape)

display(
    df_model[
        [
            "tournament_year",
            "home_team_name",
            "away_team_name",
            "text_available",
            "text_count",
            "log_text_count",
            "min_hours_before_match",
            "sources",
        ]
    ].tail(20)
)


(500, 113)


,tournament_year,home_team_name,away_team_name,text_available,text_count,log_text_count,min_hours_before_match,sources
480,2022,Korea Republic,Portugal,1,4,1.609438,6.481111,guardian
481,2022,Ghana,Uruguay,1,2,1.098612,67.995556,guardian
482,2022,Serbia,Switzerland,1,2,1.098612,83.670556,guardian
483,2022,Cameroon,Brazil,1,1,0.693147,11.748889,guardian
484,2022,Argentina,Australia,1,3,1.386294,33.144444,guardian
485,2022,Netherlands,United States,1,3,1.386294,74.486667,guardian
486,2022,England,Senegal,1,3,1.386294,10.454167,guardian
487,2022,France,Poland,1,3,1.386294,11.988056,guardian
488,2022,Japan,Croatia,1,2,1.098612,19.999444,guardian
489,2022,Brazil,Korea Republic,1,2,1.098612,50.471389,guardian


In [18]:
text_coverage_6h = (
    df_model.groupby("tournament_year")
    .agg(
        matches=("match_id", "count"),
        matches_with_text=("text_available", "sum"),
        avg_text_count=("text_count", "mean"),
    )
    .reset_index()
)

text_coverage_6h["coverage_rate"] = (
    text_coverage_6h["matches_with_text"] / text_coverage_6h["matches"]
)

display(text_coverage_6h)


,tournament_year,matches,matches_with_text,avg_text_count,coverage_rate
0,1994,52,1,0.019231,0.019231
1,1998,64,1,0.015625,0.015625
2,2002,64,57,1.843750,0.890625
3,2006,64,39,1.703125,0.609375
4,2010,64,61,2.343750,0.953125
5,2014,64,62,3.328125,0.968750
6,2018,64,51,1.812500,0.796875
7,2022,64,49,2.109375,0.765625


In [19]:
rest_day_cols = [
    "home_rest_days",
    "away_rest_days",
]

for c in rest_day_cols:
    if c in df_model.columns:
        df_model[c] = df_model[c].fillna(0)

first_match_cols = [
    "home_first_match_in_tournament",
    "away_first_match_in_tournament",
]

for c in first_match_cols:
    if c in df_model.columns:
        df_model[c] = df_model[c].fillna(False).astype(int)

display(df_model[rest_day_cols + first_match_cols].head(20))


,home_rest_days,away_rest_days,home_first_match_in_tournament,away_first_match_in_tournament
0,0.0,0.0,1,1
1,0.0,0.0,1,1
2,0.0,0.0,1,1
3,0.0,0.0,1,1
4,0.0,0.0,1,1
5,0.0,0.0,1,1
6,0.0,0.0,1,1
7,0.0,0.0,1,1
8,0.0,0.0,1,1
9,0.0,0.0,1,1


In [20]:
win_rate_cols = [c for c in df_model.columns if "win_rate" in c]

print("Win-rate columns:", len(win_rate_cols))
print(win_rate_cols)

for c in win_rate_cols:
    df_model[c] = df_model[c].fillna(0)

display(df_model[win_rate_cols].isna().sum().sort_values(ascending=False).head())


Win-rate columns: 8
['home_wc_win_rate_before', 'home_last3_win_rate', 'home_last5_win_rate', 'home_last10_win_rate', 'away_wc_win_rate_before', 'away_last3_win_rate', 'away_last5_win_rate', 'away_last10_win_rate']


home_wc_win_rate_before    0
home_last3_win_rate        0
home_last5_win_rate        0
home_last10_win_rate       0
away_wc_win_rate_before    0
dtype: int64

In [21]:
rolling_keywords = [
    "_matches_before",
    "_points_before",
    "_goals_for_before",
    "_goals_against_before",
    "_goal_diff_before",
    "_wins_before",
    "_last3_",
    "_last5_",
    "_last10_",
    "_tournament_",
]

rolling_cols = [
    c for c in df_model.columns
    if any(k in c for k in rolling_keywords)
]

print("Rolling/history columns:", len(rolling_cols))

for c in rolling_cols:
    if pd.api.types.is_numeric_dtype(df_model[c]):
        df_model[c] = df_model[c].fillna(0)

display(df_model[rolling_cols].isna().sum().sort_values(ascending=False).head(20))


Rolling/history columns: 58


home_wc_matches_before          0
away_last5_goals_for            0
away_wc_goals_for_before        0
away_wc_goals_against_before    0
away_wc_goal_diff_before        0
away_wc_wins_before             0
away_last3_matches              0
away_last3_points               0
away_last3_goals_for            0
away_last3_goals_against        0
away_last3_goal_diff            0
away_last3_win_rate             0
away_last5_matches              0
away_last5_points               0
away_last5_goals_against        0
home_wc_points_before           0
away_last5_goal_diff            0
away_last5_win_rate             0
away_last10_matches             0
away_last10_points              0
dtype: int64

In [24]:
fifa_cols = [
    c for c in df_model.columns
    if "fifa" in c and pd.api.types.is_numeric_dtype(df_model[c])
]

fifa_missing = df_model[fifa_cols].isna().mean().sort_values(ascending=False)

display(fifa_missing.head(20))

rows_with_missing_fifa = df_model[df_model[fifa_cols].isna().any(axis=1)]

print("Rows with any missing FIFA numeric feature:", len(rows_with_missing_fifa))

if len(rows_with_missing_fifa) > 0:
    display(
        rows_with_missing_fifa[
            [
                "tournament_year",
                "home_team_name",
                "away_team_name",
                "home_fifa_rank",
                "away_fifa_rank",
            ]
        ].head(50)
    )


home_fifa_rank               0.0
home_fifa_points             0.0
home_fifa_previous_points    0.0
home_fifa_diff_points        0.0
away_fifa_rank               0.0
away_fifa_points             0.0
away_fifa_previous_points    0.0
away_fifa_diff_points        0.0
fifa_rank_diff               0.0
fifa_points_diff             0.0
fifa_momentum_diff           0.0
dtype: float64

Rows with any missing FIFA numeric feature: 0


In [23]:
IMPUTE_REMAINING_FIFA_WITH_MEDIAN = True

if IMPUTE_REMAINING_FIFA_WITH_MEDIAN:
    for c in fifa_cols:
        median_value = df_model[c].median()
        df_model[c] = df_model[c].fillna(median_value)

    print("[OK] Remaining FIFA numeric NaNs imputed with median.")


[OK] Remaining FIFA numeric NaNs imputed with median.


In [25]:
# На всякий случай проверим направление признака
# Чем больше fifa_rank_diff = away_rank - home_rank, тем сильнее home team.
# EDA мог использовать home_rank - away_rank, поэтому фиксируем оба варианта.

if "home_fifa_rank" in df_model.columns and "away_fifa_rank" in df_model.columns:
    df_model["fifa_rank_diff_home_minus_away"] = (
        df_model["home_fifa_rank"] - df_model["away_fifa_rank"]
    )
    df_model["fifa_rank_diff_away_minus_home"] = (
        df_model["away_fifa_rank"] - df_model["home_fifa_rank"]
    )

if "home_fifa_points" in df_model.columns and "away_fifa_points" in df_model.columns:
    df_model["fifa_points_diff_home_minus_away"] = (
        df_model["home_fifa_points"] - df_model["away_fifa_points"]
    )

if "home_fifa_diff_points" in df_model.columns and "away_fifa_diff_points" in df_model.columns:
    df_model["fifa_momentum_diff_home_minus_away"] = (
        df_model["home_fifa_diff_points"] - df_model["away_fifa_diff_points"]
    )

if "home_better_rank_flag" in df_model.columns:
    df_model["home_better_rank_flag"] = df_model["home_better_rank_flag"].astype(int)

display(
    df_model[
        [
            "home_team_name",
            "away_team_name",
            "home_fifa_rank",
            "away_fifa_rank",
            "fifa_rank_diff_home_minus_away",
            "fifa_rank_diff_away_minus_home",
            "home_better_rank_flag",
        ]
    ].head()
)


,home_team_name,away_team_name,home_fifa_rank,away_fifa_rank,fifa_rank_diff_home_minus_away,fifa_rank_diff_away_minus_home,home_better_rank_flag
0,Germany,Bolivia,3.0,40.0,-37.0,37.0,1
1,Spain,Korea Republic,6.0,35.0,-29.0,29.0,1
2,Colombia,Romania,14.0,7.0,7.0,-7.0,0
3,Italy,Republic of Ireland,2.0,13.0,-11.0,11.0,1
4,United States,Switzerland,21.0,10.0,11.0,-11.0,0


In [26]:
pca_input_cols = [
    c for c in [
        "fifa_rank_diff_away_minus_home",
        "fifa_points_diff_home_minus_away",
        "fifa_momentum_diff_home_minus_away",
    ]
    if c in df_model.columns
]

print("PCA input cols:", pca_input_cols)

if len(pca_input_cols) >= 2:
    pca_input = df_model[pca_input_cols].copy()

    for c in pca_input_cols:
        pca_input[c] = pca_input[c].fillna(pca_input[c].median())

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(pca_input)

    pca = PCA(n_components=1, random_state=42)
    df_model["fifa_strength_pc1"] = pca.fit_transform(X_scaled)[:, 0]

    print("Explained variance ratio:", pca.explained_variance_ratio_[0])
else:
    print("[WARN] Not enough FIFA diff features for PCA.")


PCA input cols: ['fifa_rank_diff_away_minus_home', 'fifa_points_diff_home_minus_away', 'fifa_momentum_diff_home_minus_away']
Explained variance ratio: 0.7669358533282893


# Stage features

In [27]:
def normalize_stage(stage):
    if pd.isna(stage):
        return "unknown"

    s = str(stage).strip().lower()

    if "group" in s:
        return "group"

    if "round of 16" in s or "last 16" in s:
        return "round_of_16"

    if "quarter" in s:
        return "quarter_final"

    if "semi" in s:
        return "semi_final"

    if "third" in s or "3rd" in s:
        return "third_place"

    if "final" in s:
        return "final"

    return re.sub(r"\s+", "_", s)


if "stage" in df_model.columns:
    df_model["stage_norm"] = df_model["stage"].apply(normalize_stage)
else:
    df_model["stage_norm"] = "unknown"

df_model["is_knockout"] = (~df_model["stage_norm"].isin(["group", "unknown"])).astype(int)

display(df_model[["stage", "stage_norm", "is_knockout"]].drop_duplicates().head(50))


,stage,stage_norm,is_knockout
0,Group stage,group,0
36,Round of 16,round_of_16,1
44,Quarter-finals,quarter_final,1
48,Semi-finals,semi_final,1
50,Third-place match,third_place,1
51,Final,final,1


### One-hot encoding stage

In [28]:
stage_dummies = pd.get_dummies(df_model["stage_norm"], prefix="stage").astype(int)

df_model = pd.concat(
    [
        df_model.drop(columns=[c for c in stage_dummies.columns if c in df_model.columns], errors="ignore"),
        stage_dummies,
    ],
    axis=1,
)

stage_feature_cols = list(stage_dummies.columns) + ["is_knockout"]

print(stage_feature_cols)


['stage_final', 'stage_group', 'stage_quarter_final', 'stage_round_of_16', 'stage_semi_final', 'stage_third_place', 'is_knockout']


# Feature groups после EDA

In [29]:
ranking_features_full = [
    c for c in [
        "home_fifa_rank",
        "away_fifa_rank",
        "fifa_rank_diff",
        "home_fifa_points",
        "away_fifa_points",
        "fifa_points_diff",
        "home_fifa_diff_points",
        "away_fifa_diff_points",
        "fifa_momentum_diff",
        "home_better_rank_flag",
        "fifa_rank_diff_home_minus_away",
        "fifa_rank_diff_away_minus_home",
        "fifa_points_diff_home_minus_away",
        "fifa_momentum_diff_home_minus_away",
        "fifa_strength_pc1",
    ]
    if c in df_model.columns
]

ranking_features_compact = [
    c for c in [
        "fifa_rank_diff_away_minus_home",
        "home_better_rank_flag",
        "fifa_strength_pc1",
    ]
    if c in df_model.columns
]

form_last10_features = [
    c for c in df_model.columns
    if "_last10_" in c
]

form_last5_features = [
    c for c in df_model.columns
    if "_last5_" in c
]

form_last3_features = [
    c for c in df_model.columns
    if "_last3_" in c
]

wc_history_features = [
    c for c in df_model.columns
    if "_wc_" in c and (
        c.endswith("_before") or 
        "win_rate" in c or 
        "goal_diff" in c or 
        "matches" in c or 
        "points" in c
    )
]

tournament_cumulative_features = [
    c for c in df_model.columns
    if "_tournament_" in c
]

rest_features = [
    c for c in [
        "home_rest_days",
        "away_rest_days",
        "home_first_match_in_tournament",
        "away_first_match_in_tournament",
    ]
    if c in df_model.columns
]

text_meta_features = [
    c for c in [
        "text_available",
        "text_count",
        "log_text_count",
        "guardian_text_flag",
        "min_hours_before_match",
        "max_hours_before_match",
    ]
    if c in df_model.columns
]

# min/max hours can be NaN for no text. For numeric models fill later.
stage_features = [
    c for c in stage_feature_cols
    if c in df_model.columns
]

combined_numeric_features = sorted(
    set(
        ranking_features_full
        + form_last10_features
        + wc_history_features
        + tournament_cumulative_features
        + rest_features
        + text_meta_features
        + stage_features
    )
)

feature_groups = {
    "ranking_full": ranking_features_full,
    "ranking_compact": ranking_features_compact,
    "form_last10": form_last10_features,
    "form_last5": form_last5_features,
    "form_last3": form_last3_features,
    "wc_history": wc_history_features,
    "tournament_cumulative": tournament_cumulative_features,
    "rest": rest_features,
    "text_meta": text_meta_features,
    "stage": stage_features,
    "combined_numeric": combined_numeric_features,
}

for k, v in feature_groups.items():
    print(k, len(v))
    print(v[:20])
    print()


ranking_full 15
['home_fifa_rank', 'away_fifa_rank', 'fifa_rank_diff', 'home_fifa_points', 'away_fifa_points', 'fifa_points_diff', 'home_fifa_diff_points', 'away_fifa_diff_points', 'fifa_momentum_diff', 'home_better_rank_flag', 'fifa_rank_diff_home_minus_away', 'fifa_rank_diff_away_minus_home', 'fifa_points_diff_home_minus_away', 'fifa_momentum_diff_home_minus_away', 'fifa_strength_pc1']

ranking_compact 3
['fifa_rank_diff_away_minus_home', 'home_better_rank_flag', 'fifa_strength_pc1']

form_last10 12
['home_last10_matches', 'home_last10_points', 'home_last10_goals_for', 'home_last10_goals_against', 'home_last10_goal_diff', 'home_last10_win_rate', 'away_last10_matches', 'away_last10_points', 'away_last10_goals_for', 'away_last10_goals_against', 'away_last10_goal_diff', 'away_last10_win_rate']

form_last5 12
['home_last5_matches', 'home_last5_points', 'home_last5_goals_for', 'home_last5_goals_against', 'home_last5_goal_diff', 'home_last5_win_rate', 'away_last5_matches', 'away_last5_poin

In [30]:
with open(FEATURE_GROUPS_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_groups, f, ensure_ascii=False, indent=2)

print(f"[OK] Saved feature groups: {FEATURE_GROUPS_PATH}")


[OK] Saved feature groups: /Users/gaperov/Documents/University/SNA/data/processed/feature_groups.json


# Финальная numeric imputation для model-ready

In [31]:
df_model["no_text_flag"] = (df_model["text_available"] == 0).astype(int)

for c in ["min_hours_before_match", "max_hours_before_match"]:
    if c in df_model.columns:
        df_model[c] = df_model[c].fillna(0)

if "no_text_flag" not in feature_groups["text_meta"]:
    feature_groups["text_meta"].append("no_text_flag")

if "no_text_flag" not in feature_groups["combined_numeric"]:
    feature_groups["combined_numeric"].append("no_text_flag")


In [32]:
leakage_cols = {
    "home_score",
    "away_score",
    "result",
    "target",
    "home_xg",
    "away_xg",
    "xg",
}

for group_name, cols in feature_groups.items():
    feature_groups[group_name] = [
        c for c in cols
        if c not in leakage_cols
    ]

combined_numeric_features = feature_groups["combined_numeric"]

print("Combined numeric features:", len(combined_numeric_features))
print("Leakage cols present:", [c for c in combined_numeric_features if c in leakage_cols])


Combined numeric features: 69
Leakage cols present: []


In [33]:
for c in combined_numeric_features:
    if c not in df_model.columns:
        continue

    if pd.api.types.is_bool_dtype(df_model[c]):
        df_model[c] = df_model[c].astype(int)

    if pd.api.types.is_numeric_dtype(df_model[c]):
        if df_model[c].isna().any():
            df_model[c] = df_model[c].fillna(0)

missing_combined = df_model[combined_numeric_features].isna().sum()
display(missing_combined[missing_combined > 0])


Series([], dtype: int64)

# Class weights

In [34]:
target_col = "target"

classes = np.sort(df_model[target_col].dropna().unique()).astype(int)

computed_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=df_model[target_col].astype(int),
)

computed_class_weights = {
    int(cls): float(w)
    for cls, w in zip(classes, computed_weights)
}

eda_recommended_class_weights = {
    0: 1.00,  # home_win
    1: 1.86,  # draw
    2: 1.35,  # away_win
}

class_weights_payload = {
    "target_mapping": {
        "0": "home_win",
        "1": "draw",
        "2": "away_win",
    },
    "computed_balanced": computed_class_weights,
    "eda_recommended": eda_recommended_class_weights,
}

display(class_weights_payload)

with open(CLASS_WEIGHTS_PATH, "w", encoding="utf-8") as f:
    json.dump(class_weights_payload, f, indent=2, ensure_ascii=False)

print(f"[OK] Saved class weights: {CLASS_WEIGHTS_PATH}")


{'target_mapping': {'0': 'home_win', '1': 'draw', '2': 'away_win'},
 'computed_balanced': {0: 0.76103500761035,
  1: 1.4124293785310735,
  2: 1.0224948875255624},
 'eda_recommended': {0: 1.0, 1: 1.86, 2: 1.35}}

[OK] Saved class weights: /Users/gaperov/Documents/University/SNA/data/processed/class_weights.json


# Model-ready split

In [35]:
def split_by_tournament_year(
    df,
    train_years=(1994, 1998, 2002, 2006, 2010, 2014),
    val_years=(2018,),
    test_years=(2022,),
):
    train = df[df["tournament_year"].isin(train_years)].copy()
    val = df[df["tournament_year"].isin(val_years)].copy()
    test = df[df["tournament_year"].isin(test_years)].copy()

    return train, val, test


train_model, val_model, test_model = split_by_tournament_year(df_model)

print("train:", train_model.shape)
print("val:", val_model.shape)
print("test:", test_model.shape)

display(train_model["result"].value_counts(normalize=True))
display(val_model["result"].value_counts(normalize=True))
display(test_model["result"].value_counts(normalize=True))


train: (372, 127)
val: (64, 127)
test: (64, 127)


result
home_win    0.440860
away_win    0.317204
draw        0.241935
Name: proportion, dtype: float64

result
home_win    0.406250
away_win    0.390625
draw        0.203125
Name: proportion, dtype: float64

result
home_win    0.453125
away_win    0.312500
draw        0.234375
Name: proportion, dtype: float64

In [36]:
save_table(df_model, PROCESSED_DIR / "match_dataset_model_ready.parquet")

save_table(train_model, PROCESSED_DIR / "train_matches_model_ready.parquet")
save_table(val_model, PROCESSED_DIR / "val_matches_model_ready.parquet")
save_table(test_model, PROCESSED_DIR / "test_matches_model_ready.parquet")

# Пересохраняем feature_groups после добавления no_text_flag
with open(FEATURE_GROUPS_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_groups, f, ensure_ascii=False, indent=2)


[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/match_dataset_model_ready.parquet
[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/train_matches_model_ready.parquet
[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/val_matches_model_ready.parquet
[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/test_matches_model_ready.parquet


In [37]:
selected_feature_group = "combined_numeric"
selected_features = feature_groups[selected_feature_group]

X_train = train_model[selected_features].copy()
y_train = train_model["target"].astype(int)

X_val = val_model[selected_features].copy()
y_val = val_model["target"].astype(int)

X_test = test_model[selected_features].copy()
y_test = test_model["target"].astype(int)

print("Selected feature group:", selected_feature_group)
print("n_features:", len(selected_features))
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

display(X_train.head())


Selected feature group: combined_numeric
n_features: 69
X_train: (372, 69)
X_val: (64, 69)
X_test: (64, 69)


,away_fifa_diff_points,away_fifa_points,away_fifa_rank,away_first_match_in_tournament,away_last10_goal_diff,away_last10_goals_against,away_last10_goals_for,away_last10_matches,away_last10_points,away_last10_win_rate,...,min_hours_before_match,stage_final,stage_group,stage_quarter_final,stage_round_of_16,stage_semi_final,stage_third_place,text_available,text_count,no_text_flag
0,1.0,36.0,40.0,1,-16,16,0,3,0,0.0,...,0.0,0,1,0,0,0,0,0,0,1
1,1.0,39.0,35.0,1,-24,29,5,8,1,0.0,...,0.0,0,1,0,0,0,0,0,0,1
2,3.0,58.0,7.0,1,-2,15,13,10,9,0.2,...,0.0,0,1,0,0,0,0,0,0,1
3,1.0,54.0,13.0,1,-1,3,2,5,4,0.0,...,0.0,0,1,0,0,0,0,0,0,1
4,2.0,56.0,10.0,1,-14,28,14,10,6,0.2,...,0.0,0,1,0,0,0,0,0,0,1


# Изменения перед созданием графов

### Обновленный код построения model-ready graph

In [38]:
def make_hash_id(prefix, *parts, length=12):
    raw = "||".join("" if p is None else str(p) for p in parts)
    import hashlib
    h = hashlib.md5(raw.encode("utf-8")).hexdigest()[:length]
    return f"{prefix}_{h}"


def normalize_text_name(x):
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[^\w\s]", "", x)
    return x.strip()


def build_model_ready_graph_nodes(df):
    nodes = []

    # Team nodes
    team_rows = []

    for side in ["home", "away"]:
        team_rows.append(
            df[
                [
                    f"{side}_team_id",
                    f"{side}_team_name",
                    f"{side}_team_norm",
                ]
            ].rename(
                columns={
                    f"{side}_team_id": "team_id",
                    f"{side}_team_name": "team_name",
                    f"{side}_team_norm": "team_norm",
                }
            )
        )

    teams_df = pd.concat(team_rows, ignore_index=True).drop_duplicates("team_id")

    for _, r in teams_df.iterrows():
        nodes.append(
            {
                "node_id": r["team_id"],
                "node_type": "Team",
                "name": r["team_name"],
                "canonical_name": r["team_norm"],
                "source_id": None,
                "tournament_year": None,
                "metadata_json": json.dumps(
                    {
                        "team_norm": r["team_norm"],
                    },
                    ensure_ascii=False,
                ),
            }
        )

    # Match nodes
    for _, r in df.drop_duplicates("match_id").iterrows():
        nodes.append(
            {
                "node_id": r["match_id"],
                "node_type": "Match",
                "name": f"{r['home_team_name']} vs {r['away_team_name']} {r['tournament_year']}",
                "canonical_name": r["match_id"],
                "source_id": r["match_id"],
                "tournament_year": int(r["tournament_year"]),
                "metadata_json": json.dumps(
                    {
                        "match_date": str(r.get("match_date")),
                        "stage": r.get("stage"),
                        "stage_norm": r.get("stage_norm"),
                        "is_knockout": int(r.get("is_knockout", 0)),
                        "text_available": int(r.get("text_available", 0)),
                        "text_count": int(r.get("text_count", 0)),
                        "log_text_count": float(r.get("log_text_count", 0)),
                        "target": None if pd.isna(r.get("target")) else int(r.get("target")),
                        "result": r.get("result"),
                    },
                    ensure_ascii=False,
                ),
            }
        )

    # Tournament nodes
    for year in sorted(df["tournament_year"].dropna().astype(int).unique()):
        nodes.append(
            {
                "node_id": f"wc_{year}",
                "node_type": "Tournament",
                "name": f"FIFA World Cup {year}",
                "canonical_name": f"world cup {year}",
                "source_id": f"wc_{year}",
                "tournament_year": int(year),
                "metadata_json": "{}",
            }
        )

    # Stadium nodes
    if "stadium_id" in df.columns:
        stadiums = (
            df[["stadium_id", "stadium_name"]]
            .dropna(subset=["stadium_id"])
            .drop_duplicates("stadium_id")
        )

        for _, r in stadiums.iterrows():
            nodes.append(
                {
                    "node_id": r["stadium_id"],
                    "node_type": "Stadium",
                    "name": r["stadium_name"],
                    "canonical_name": normalize_text_name(r["stadium_name"]),
                    "source_id": None,
                    "tournament_year": None,
                    "metadata_json": "{}",
                }
            )

    # Referee nodes
    if "referee_id" in df.columns:
        refs = (
            df[["referee_id", "referee_name"]]
            .dropna(subset=["referee_id"])
            .drop_duplicates("referee_id")
        )

        for _, r in refs.iterrows():
            nodes.append(
                {
                    "node_id": r["referee_id"],
                    "node_type": "Referee",
                    "name": r["referee_name"],
                    "canonical_name": normalize_text_name(r["referee_name"]),
                    "source_id": None,
                    "tournament_year": None,
                    "metadata_json": "{}",
                }
            )

    # TeamTournament nodes
    tt_rows = []

    for _, r in df.iterrows():
        for side in ["home", "away"]:
            tt_rows.append(
                {
                    "team_id": r[f"{side}_team_id"],
                    "team_name": r[f"{side}_team_name"],
                    "team_norm": r[f"{side}_team_norm"],
                    "tournament_id": r["tournament_id"],
                    "tournament_year": r["tournament_year"],
                    "fifa_rank": r.get(f"{side}_fifa_rank"),
                    "fifa_points": r.get(f"{side}_fifa_points"),
                    "wc_matches_before": r.get(f"{side}_wc_matches_before"),
                    "wc_win_rate_before": r.get(f"{side}_wc_win_rate_before"),
                }
            )

    tt_df = pd.DataFrame(tt_rows).drop_duplicates(["team_id", "tournament_id"])

    for _, r in tt_df.iterrows():
        tt_id = make_hash_id("team_tournament", r["team_id"], r["tournament_id"])

        nodes.append(
            {
                "node_id": tt_id,
                "node_type": "TeamTournament",
                "name": f"{r['team_name']} {int(r['tournament_year'])}",
                "canonical_name": tt_id,
                "source_id": None,
                "tournament_year": int(r["tournament_year"]),
                "metadata_json": json.dumps(
                    {
                        "team_id": r["team_id"],
                        "team_norm": r["team_norm"],
                        "tournament_id": r["tournament_id"],
                        "fifa_rank": None if pd.isna(r["fifa_rank"]) else float(r["fifa_rank"]),
                        "fifa_points": None if pd.isna(r["fifa_points"]) else float(r["fifa_points"]),
                        "wc_matches_before": None if pd.isna(r["wc_matches_before"]) else float(r["wc_matches_before"]),
                        "wc_win_rate_before": None if pd.isna(r["wc_win_rate_before"]) else float(r["wc_win_rate_before"]),
                    },
                    ensure_ascii=False,
                ),
            }
        )

    return pd.DataFrame(nodes).drop_duplicates("node_id").reset_index(drop=True)


In [39]:
def build_model_ready_graph_edges(df):
    edges = []

    for _, r in df.iterrows():
        match_id = r["match_id"]
        tournament_id = r["tournament_id"]
        tournament_year = int(r["tournament_year"])

        # Match -> Tournament
        edges.append(
            {
                "edge_id": make_hash_id("edge", "match_belongs_to_tournament", match_id, tournament_id),
                "source_node_id": match_id,
                "target_node_id": tournament_id,
                "edge_type": "match_belongs_to_tournament",
                "match_id": match_id,
                "tournament_id": tournament_id,
                "tournament_year": tournament_year,
                "timestamp": r["match_date"],
                "weight": 1.0,
                "metadata_json": "{}",
            }
        )

        # Match -> Stadium
        if pd.notna(r.get("stadium_id")):
            edges.append(
                {
                    "edge_id": make_hash_id("edge", "match_played_at_stadium", match_id, r["stadium_id"]),
                    "source_node_id": match_id,
                    "target_node_id": r["stadium_id"],
                    "edge_type": "match_played_at_stadium",
                    "match_id": match_id,
                    "tournament_id": tournament_id,
                    "tournament_year": tournament_year,
                    "timestamp": r["match_date"],
                    "weight": 1.0,
                    "metadata_json": "{}",
                }
            )

        # Referee -> Match
        if pd.notna(r.get("referee_id")):
            edges.append(
                {
                    "edge_id": make_hash_id("edge", "referee_officiated_match", r["referee_id"], match_id),
                    "source_node_id": r["referee_id"],
                    "target_node_id": match_id,
                    "edge_type": "referee_officiated_match",
                    "match_id": match_id,
                    "tournament_id": tournament_id,
                    "tournament_year": tournament_year,
                    "timestamp": r["match_date"],
                    "weight": 1.0,
                    "metadata_json": "{}",
                }
            )

        for side in ["home", "away"]:
            opp_side = "away" if side == "home" else "home"

            team_id = r[f"{side}_team_id"]
            opp_team_id = r[f"{opp_side}_team_id"]

            tt_id = make_hash_id("team_tournament", team_id, tournament_id)

            # Team -> Match with pre-match side features
            side_metadata = {
                "side": side,
                "team_fifa_rank": None if pd.isna(r.get(f"{side}_fifa_rank")) else float(r.get(f"{side}_fifa_rank")),
                "team_fifa_points": None if pd.isna(r.get(f"{side}_fifa_points")) else float(r.get(f"{side}_fifa_points")),
                "team_wc_matches_before": float(r.get(f"{side}_wc_matches_before", 0)),
                "team_wc_win_rate_before": float(r.get(f"{side}_wc_win_rate_before", 0)),
                "team_last10_win_rate": float(r.get(f"{side}_last10_win_rate", 0)),
                "team_last10_goal_diff": float(r.get(f"{side}_last10_goal_diff", 0)),
                "team_rest_days": float(r.get(f"{side}_rest_days", 0)),
                "team_first_match_in_tournament": int(r.get(f"{side}_first_match_in_tournament", 0)),
            }

            edges.append(
                {
                    "edge_id": make_hash_id("edge", "team_played_match", team_id, match_id, side),
                    "source_node_id": team_id,
                    "target_node_id": match_id,
                    "edge_type": "team_played_match",
                    "match_id": match_id,
                    "tournament_id": tournament_id,
                    "tournament_year": tournament_year,
                    "timestamp": r["match_date"],
                    "weight": 1.0,
                    "metadata_json": json.dumps(side_metadata, ensure_ascii=False),
                }
            )

            # Team -> Opponent Team
            edges.append(
                {
                    "edge_id": make_hash_id("edge", "team_played_against_team", team_id, opp_team_id, match_id),
                    "source_node_id": team_id,
                    "target_node_id": opp_team_id,
                    "edge_type": "team_played_against_team",
                    "match_id": match_id,
                    "tournament_id": tournament_id,
                    "tournament_year": tournament_year,
                    "timestamp": r["match_date"],
                    "weight": 1.0,
                    "metadata_json": json.dumps(
                        {
                            "side": side,
                            "stage_norm": r.get("stage_norm"),
                            "is_knockout": int(r.get("is_knockout", 0)),
                        },
                        ensure_ascii=False,
                    ),
                }
            )

            # Team -> Tournament
            edges.append(
                {
                    "edge_id": make_hash_id("edge", "team_participated_in_tournament", team_id, tournament_id),
                    "source_node_id": team_id,
                    "target_node_id": tournament_id,
                    "edge_type": "team_participated_in_tournament",
                    "match_id": None,
                    "tournament_id": tournament_id,
                    "tournament_year": tournament_year,
                    "timestamp": None,
                    "weight": 1.0,
                    "metadata_json": "{}",
                }
            )

            # Team -> TeamTournament
            edges.append(
                {
                    "edge_id": make_hash_id("edge", "team_has_tournament_instance", team_id, tt_id),
                    "source_node_id": team_id,
                    "target_node_id": tt_id,
                    "edge_type": "team_has_tournament_instance",
                    "match_id": None,
                    "tournament_id": tournament_id,
                    "tournament_year": tournament_year,
                    "timestamp": None,
                    "weight": 1.0,
                    "metadata_json": "{}",
                }
            )

            # TeamTournament -> Match
            edges.append(
                {
                    "edge_id": make_hash_id("edge", "team_tournament_played_match", tt_id, match_id, side),
                    "source_node_id": tt_id,
                    "target_node_id": match_id,
                    "edge_type": "team_tournament_played_match",
                    "match_id": match_id,
                    "tournament_id": tournament_id,
                    "tournament_year": tournament_year,
                    "timestamp": r["match_date"],
                    "weight": 1.0,
                    "metadata_json": json.dumps(side_metadata, ensure_ascii=False),
                }
            )

    return pd.DataFrame(edges).drop_duplicates("edge_id").reset_index(drop=True)


In [40]:
graph_nodes_model_ready = build_model_ready_graph_nodes(df_model)
graph_edges_model_ready = build_model_ready_graph_edges(df_model)

print("Nodes:", graph_nodes_model_ready.shape)
print("Edges:", graph_edges_model_ready.shape)

display(graph_nodes_model_ready["node_type"].value_counts())
display(graph_edges_model_ready["edge_type"].value_counts())

save_table(graph_nodes_model_ready, PROCESSED_DIR / "graph_nodes_model_ready.parquet")
save_table(graph_edges_model_ready, PROCESSED_DIR / "graph_edges_model_ready.parquet")


Nodes: (1007, 7)
Edges: (4741, 10)


node_type
Match             500
TeamTournament    248
Stadium            93
Referee            88
Team               70
Tournament          8
Name: count, dtype: int64

edge_type
team_played_match                  1000
team_played_against_team           1000
team_tournament_played_match       1000
match_belongs_to_tournament         500
match_played_at_stadium             500
team_participated_in_tournament     248
team_has_tournament_instance        248
referee_officiated_match            245
Name: count, dtype: int64

[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/graph_nodes_model_ready.parquet
[OK] Saved parquet: /Users/gaperov/Documents/University/SNA/data/processed/graph_edges_model_ready.parquet
